# Secure Data Disclosure: Client side

This notebook showcases how researcher could use the Secure Data Disclosure system. It explains the different functionnalities provided by the `lomas_client` library to interact with the secure server.

The secure data are never visible by researchers. They can only access to differentially private responses via queries to the server.

Each user has access to one or multiple projects and for each dataset has a limited budget with $\epsilon$ and $\delta$ values.

In [ ]:
from rich.jupyter import print

%load_ext rich

The rich extension is already loaded. To reload it, use:
  %reload_ext rich


In [ ]:
# from IPython.display import Image
# Image(filename="images/image_demo_client.png", width=800)

We will use a synthetic dataset about COVID to demonstrate the how to use the library `lomas_client` with polars queries.

## Step 1: Install the library

It can be installed via the pip command:

In [ ]:
import sys
import os
sys.path.append(os.path.abspath(os.path.join('..')))
# !pip install lomas_client

In [ ]:
from lomas_client import Client
import numpy as np
import opendp.prelude as dp

## Step 2: Initialise the client

Once the library is installed, a Client object must be created. It is responsible for sending sending requests to the server and processing responses in the local environment. It enables a seamless interaction with the server. 

The client needs a few parameters to be created. Usually, these would be set in the environment by the system administrator and be transparent to lomas users. In this instance, the following code snippet sets a few of these parameters that are specific to this notebook. 

In [ ]:
# The following would usually be set in the environment by a system administrator
# and be tranparent to lomas users.
# Uncomment them if you are running against a Kubernetes deployment.
# They have already been set for you if you are running locally within a devenv or the Jupyter lab set up by Docker compose.

import os
# os.environ["LOMAS_CLIENT_APP_URL"] = "https://lomas.example.com:443"
# os.environ["LOMAS_CLIENT_KEYCLOAK_URL"] = "https://keycloak.example.com:443"
# os.environ["LOMAS_CLIENT_TELEMETRY__ENABLED"] = "false"
# os.environ["LOMAS_CLIENT_TELEMETRY__COLLECTOR_ENDPOINT"] = "http://otel.example.com:445"
# os.environ["LOMAS_CLIENT_TELEMETRY__COLLECTOR_INSECURE"] = "true"
# os.environ["LOMAS_CLIENT_TELEMETRY__SERVICE_ID"] = "my-app-client"
# os.environ["LOMAS_CLIENT_REALM"] = "lomas"

# We set these ones because they are specific to this notebook.

USER_NAME = "Mr.Corona"
os.environ["LOMAS_CLIENT_CLIENT_ID"] = USER_NAME
os.environ["LOMAS_CLIENT_CLIENT_SECRET"] = USER_NAME.lower()
os.environ["LOMAS_CLIENT_DATASET_NAME"] = "COVID_SYNTHETIC"

# Note that all client settings can also be passed as keyword arguments to the Client constructor.
# eg. client = Client(client_id = "Dr.Antartica") takes precedence over setting the "LOMAS_CLIENT_CLIENT_ID"
# environment variable.

In [ ]:
client = Client()

[11:01:54] WARNING  Keycloak or Lomas service configured without TLS -> using oauthlib insecure   ]8;id=939297;file:///home/azureuser/Desktop/POC/lomas/client/lomas_client/http_client.py\http_client.py]8;;\:]8;id=134550;file:///home/azureuser/Desktop/POC/lomas/client/lomas_client/http_client.py#30\30]8;;\
                    transport                                                                                      

## Step 3: Metadata and dummy dataset

### Getting dataset metadata

The user has never seen the data and as a first step to understand what is available to her, she would like to check the metadata of the dataset. Therefore, she just needs to call the `get_dataset_metadata()` function of the client. As this is public information, this does not cost any budget.

This function returns metadata information in a format based on [SmartnoiseSQL dictionary format](https://docs.smartnoise.org/sql/metadata.html#dictionary-format), where among other, there is information about all the available columns, their type, bound values (see Smartnoise page for more details). Any metadata is required for Smartnoise-SQL is also required here and additional information such that the different categories in a string type column column can be added.

In [ ]:
covid_metadata = client.get_dataset_metadata()
covid_metadata

### Get a dummy dataset

Now, that she has seen and understood the metadata, she wants to get an even better understanding of the dataset (but is still not able to see it). A solution to have an idea of what the dataset looks like it to create a dummy dataset. 

Based on the public metadata of the dataset, a random dataframe can be created created. By default, there will be 100 rows and the seed is set to 42 to ensure reproducibility, but these 2 variables can be changed to obtain different dummy datasets.
Getting a dummy dataset does not affect the budget as there is no differential privacy here. It is not a synthetic dataset and all that could be learn here is already present in the public metadata (it is created randomly on the fly based on the metadata).

Dr. FSO first create a dummy dataset with 200 rows and chooses a seed of 0.

In [ ]:
NB_ROWS = 200
SEED = 0

In [ ]:
dummy_lf = client.get_dummy_dataset(nb_rows=NB_ROWS, seed=SEED, lazy=True)
print(dummy_lf.collect())

shape: (200, 13)
┌────────────┬──────┬──────────────┬──────────┬───┬─────────┬─────────┬─────────────────┬───────┐
│ patient_id ┆ id   ┆ date         ┆ temporal ┆ … ┆ country ┆ subType ┆ hospitalization ┆ death │
│ ---        ┆ ---  ┆ ---          ┆ ---      ┆   ┆ ---     ┆ ---     ┆ ---             ┆ ---   │
│ i32        ┆ i32  ┆ datetime[ns] ┆ i32      ┆   ┆ str     ┆ str     ┆ bool            ┆ bool  │
╞════════════╪══════╪══════════════╪══════════╪═══╪═════════╪═════════╪═════════════════╪═══════╡
│ 42572      ┆ 460  ┆ 2023-07-18   ┆ 31       ┆ … ┆ CH      ┆ unknown ┆ false           ┆ true  │
│            ┆      ┆ 00:00:00     ┆          ┆   ┆         ┆         ┆                 ┆       │
│ 31878      ┆ 960  ┆ 2022-11-25   ┆ 47       ┆ … ┆ unknown ┆ BA.1    ┆ true            ┆ true  │
│            ┆      ┆ 00:00:00     ┆          ┆   ┆         ┆         ┆                 ┆       │
│ 25581      ┆ 654  ┆ 2023-05-18   ┆ 34       ┆ … ┆ unknown ┆ BQ.1    ┆ false           ┆ true  │
│            ┆      ┆ 00:00:00     ┆          ┆   ┆         ┆         ┆                 ┆       │
│ 13502      ┆ 465  ┆ 2022-10-08   ┆ 31       ┆ … ┆ other   ┆ XBB     ┆ false           ┆ true  │
│            ┆      ┆ 00:00:00     ┆          ┆   ┆         ┆         ┆                 ┆       │
│ 15406      ┆ 1786 ┆ 2022-12-10   ┆ 26       ┆ … ┆ FL      ┆ unknown ┆ false           ┆ true  │
│            ┆      ┆ 00:00:00     ┆          ┆   ┆         ┆         ┆                 ┆       │
│ …          ┆ …    ┆ …            ┆ …        ┆ … ┆ …       ┆ …       ┆ …               ┆ …     │
│ 48678      ┆ 1546 ┆ 2023-07-06   ┆ 5        ┆ … ┆ FL      ┆ XBB     ┆ true            ┆ false │
│            ┆      ┆ 00:00:00     ┆          ┆   ┆         ┆         ┆                 ┆       │
│ 18378      ┆ 1484 ┆ 2022-11-16   ┆ 45       ┆ … ┆ CH      ┆ BA.5    ┆ false           ┆ true  │
│            ┆      ┆ 00:00:00     ┆          ┆   ┆         ┆         ┆                 ┆       │
│ 44539      ┆ 1957 ┆ 2023-04-06   ┆ 23       ┆ … ┆ CH      ┆ BA.4    ┆ true            ┆ false │
│            ┆      ┆ 00:00:00     ┆          ┆   ┆         ┆         ┆                 ┆       │
│ 19210      ┆ 1059 ┆ 2022-12-06   ┆ 50       ┆ … ┆ FL      ┆ BA.2.75 ┆ false           ┆ true  │
│            ┆      ┆ 00:00:00     ┆          ┆   ┆         ┆         ┆                 ┆       │
│ 41158      ┆ 1180 ┆ 2023-05-27   ┆ 21       ┆ … ┆ FL      ┆ BA.5    ┆ true            ┆ false │
│            ┆      ┆ 00:00:00     ┆          ┆   ┆         ┆         ┆                 ┆       │
└────────────┴──────┴──────────────┴──────────┴───┴─────────┴─────────┴─────────────────┴───────┘

In [ ]:
test = client.get_dummy_dataset(nb_rows=NB_ROWS, seed = SEED)
test.dtypes


patient_id                  int32
id                          int32
date               datetime64[ns]
temporal                    int32
georegion          string[python]
agegroup           string[python]
sex                string[python]
testType           string[python]
testResult         string[python]
country            string[python]
subType            string[python]
hospitalization           boolean
death                     boolean
dtype: object

In [ ]:
context = client.get_context(nb_rows=NB_ROWS, seed=SEED, epsilon=1.0)

In [ ]:
context

## Step 4: Prepare the pipeline

It is necessary to prepare the pipeline before sending the query to the client.

In [ ]:
import polars as pl
pl.__name__, pl.__version__

('polars', '1.32.0')

* basic computations (mean, sum, etc.)

* basic on dates

* group by / agg

* with_columns (row-wise)

* join

* drop

* filter / select

* sort


### basic computation

a. Dataframe length

In [ ]:
plan = context.query().select(dp.len())

In [ ]:
plan.release().collect()

len
u32
200


In [ ]:
res = client.opendp.query(plan, epsilon=1.0)
print(res.result.value)

shape: (1, 1)
┌───────┐
│ len   │
│ ---   │
│ i64   │
╞═══════╡
│ 50048 │
└───────┘

b. sum

In [ ]:
context = client.get_context(nb_rows=NB_ROWS, seed=SEED, epsilon=1.0)

In [ ]:
plan = context.query().select(pl.col("temporal").dp.sum(bounds=(1,52)))

In [ ]:
# result on dummy dataset with dp
plan.release().collect()

temporal
i32
5273


In [ ]:
# result on dummy dataset without dp
dummy_lf.select(pl.col("temporal").sum()).collect()

temporal
i32
5365


In [ ]:
# actual result on remote server (with DP)
res = client.opendp.query(plan, epsilon=1.0)
print(res.result.value)

shape: (1, 1)
┌──────────┐
│ temporal │
│ ---      │
│ i64      │
╞══════════╡
│ 1327337  │
└──────────┘

### Aggregation

In [ ]:
context = client.get_context(nb_rows=NB_ROWS, seed=SEED, epsilon=1.0, delta=1e-06)

In [ ]:
# Count the number of death per sex and age_group
plan = (
    context.query()
    .group_by(["sex", "agegroup"])
    .agg([
        pl.col("death").cast(int).dp.sum(bounds=(0,1))
    ])
)
result = plan.release().collect()
result.sort("death", descending=True)

sex,agegroup,death
str,str,i64
"""other""","""20 - 29""",13
"""unknown""","""70 - 79""",12
"""other""","""10 - 19""",11
"""other""","""60 - 69""",8
"""male""","""0 - 9""",7
…,…,…
"""male""","""80+""",-4
"""unknown""","""20 - 29""",-7
"""female""","""50 - 59""",-8


### `with_columns`

In [ ]:
context = client.get_context(nb_rows=NB_ROWS, seed=SEED, epsilon=1.0, delta=1e-06)

[13:49:17] WARNING  epsilon should be less than or equal to 5, and is typically less than or equal   ]8;id=312281;file:///home/azureuser/Desktop/POC/lomas/.devenv/state/venv/lib/python3.13/site-packages/opendp/context.py\context.py]8;;\:]8;id=301039;file:///home/azureuser/Desktop/POC/lomas/.devenv/state/venv/lib/python3.13/site-packages/opendp/context.py#311\311]8;;\
                    to 1                                                                                           

In [ ]:
breaks = [13, 26, 39]
labels = pl.Series("quarter", list(range(len(breaks) + 1)), dtype=pl.UInt32)
plan = (
    context.query()
    .with_columns(
        pl.col.temporal.cut(
            breaks=breaks, left_closed=True
        ).to_physical().alias("quarter")
    )
    .group_by(pl.col.quarter)
    .agg([
        pl.col("death").cast(int).dp.sum(bounds=(0,1))
    ])
    .with_keys(pl.LazyFrame([labels]))
)
release = plan.release()

In [ ]:
release.collect().sort("quarter")

quarter,death
u32,i64
0,30
1,24
2,21
3,18


In [ ]:
res = client.opendp.query(plan, epsilon = 1.0, delta = 1e-6)
res.result.value.sort("quarter")

quarter,death
i64,i64
0,-3
1,23
2,17
3,3


### Datetime

a. Group by `year`

In [ ]:
context = client.get_context(nb_rows=NB_ROWS, seed=SEED, epsilon=1.0, delta=1e-06)

In [ ]:
# Group by year and count the number of rows per year
plan = (
    context.query()
    .with_columns(YEAR=pl.col.date.dt.year(), MONTH=pl.col.date.dt.month())
    .group_by("YEAR")
    .agg(dp.len())
)

In [ ]:
# Check plan works on local dummy dataset
plan.release().collect()

YEAR,len
i32,u32
2023,121
2022,90


In [ ]:
# apply plan on remote server (private data)
res = client.opendp.query(plan, epsilon=1.0, delta=1e-06)
print(res.result.value)

shape: (2, 2)
┌──────┬───────┐
│ YEAR ┆ len   │
│ ---  ┆ ---   │
│ i64  ┆ i64   │
╞══════╪═══════╡
│ 2023 ┆ 25146 │
│ 2022 ┆ 24899 │
└──────┴───────┘

b. Group by "month"

In [ ]:
context = client.get_context(nb_rows=NB_ROWS, seed=SEED, epsilon=1.0, delta=1e-06)
plan = (
    context.query()
    .with_columns(YEAR=pl.col.date.dt.year(), MONTH=pl.col.date.dt.month())
    .group_by("MONTH")
    .agg(dp.len())
)

# apply plan on remote server (private data)
res = client.opendp.query(plan, epsilon=1.0, delta=1e-06)
print(res.result.value.sort("MONTH"))

shape: (12, 2)
┌───────┬──────┐
│ MONTH ┆ len  │
│ ---   ┆ ---  │
│ i64   ┆ i64  │
╞═══════╪══════╡
│ 1     ┆ 8295 │
│ 2     ┆ 6932 │
│ 3     ┆ 6185 │
│ 4     ┆ 2997 │
│ 5     ┆ 274  │
│ …     ┆ …    │
│ 8     ┆ 1374 │
│ 9     ┆ 2409 │
│ 10    ┆ 5375 │
│ 11    ┆ 7322 │
│ 12    ┆ 8414 │
└───────┴──────┘

### `filter`

a. basic filter

In [ ]:
context = client.get_context(nb_rows=NB_ROWS, seed=SEED, epsilon=1.0, delta=1e-06)

In [ ]:
plan = (
    context.query()
    .filter(pl.col.sex == "female")
    .filter(pl.col("temporal") >= 26)
    .select(
        pl.col.death.cast(int)
        .dp.sum(bounds=(0,1))
    )
)
release = plan.release()

In [ ]:
# dummy result with dp
release.collect()

death
i64
12


In [ ]:
# dummy result without dp

(dummy_lf
    .filter(pl.col.sex == "female")
    .filter(pl.col("temporal") >= 26)
    .select(
        pl.col.death
        .sum()
    )
).collect()

death
u32
12


In [ ]:
res = client.opendp.query(plan, epsilon=1.0, delta=1e-6)
res.result.value

death
i64
14


2. `n_unique`

In [ ]:
context = client.get_context(nb_rows=NB_ROWS, seed=SEED, epsilon=1.0, delta=1e-06)

In [ ]:
plan = context.query().select(pl.col.id.dp.n_unique())
plan.release().collect().item() 

191

### `join`

In [ ]:
context = client.get_context(nb_rows=NB_ROWS, seed=SEED, epsilon=1.0, delta=1e-06)

In [ ]:
breaks = [13, 26, 39]
labels = pl.LazyFrame(pl.Series("quarter", list(range(len(breaks) + 1)), dtype=pl.UInt32))

In [ ]:
plan = (
    context.query()
    .with_columns(
        pl.col.temporal.cut(
            breaks=breaks, left_closed=True
        ).to_physical().alias("quarter")
    )
    .group_by(pl.col.quarter)
    .agg([
        pl.col("death").cast(int).dp.sum(bounds=(0,1))
    ])
    .join(labels, how="right", on="quarter")
)
collect = plan.release().collect()

In [ ]:
collect.sort("quarter")

death,quarter
i64,u32
24,0
14,1
22,2
19,3


In [ ]:
res = client.opendp.query(plan, epsilon=1.0, delta=1e-6)
res.result.value.sort("quarter")

death,quarter
i64,i64
5,0
30,1
13,2
1,3
